In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
import camb
import h5py
import healpy as hp
import numpy as np
from astropy import units as u
from ksw import KSW, Cosmology, Data, Shape
from mpi4py import MPI

sys.path.append(os.path.join(os.getcwd(), "scripts"))
from utils import Config, save_data, setup_logging
from utils.plots import plot_ksw_predictions

comm = MPI.COMM_WORLD
rank = comm.Get_rank()

In [ ]:
def beam(alm):
    if beam_width_rad == 0:
        return alm

    return hp.sphtfunc.smoothalm(alm, fwhm=beam_width_rad, inplace=False)

In [ ]:
s = Config(
    [
        "settings/l500_n128.json",
        "--nsims",
        "200",
        "--narray",
        "500",
        "--disable_lensing",
        "--disable_noise",
    ]
)

# s = Config(
#     ["settings/l500_1000.json", "--nsims", "1000", "--narray", "100", "--disable_noise"]
# )

camb_params_obj = camb.set_params(**s.cosmo_params)
cosmo = Cosmology(camb_params_obj)
cosmo.compute_transfer(s.cosmo_params["max_l"])
cosmo.compute_c_ell()

# create the local shape
loc_shape = Shape.prim_local(s.cosmo_params["ns"], s.cosmo_params["pivot_scalar"])
cosmo.add_prim_reduced_bispectrum(loc_shape, s.radii)

# setup the data and get our icov object
data = Data(s.lmax, s.noise_ell, s.beam_ell, s.pols, cosmo)
icov = data.icov_diag_lensed if s.lensing else data.icov_diag_nonlensed

# generate our beam functioned based on noise
beam_width_rad = 0 if s.disable_noise else s.beam_width.to_value(u.radian)

ksw = KSW(
    cosmo.red_bispectra,
    icov,
    beam,
    s.lmax,
    s.pols,
    precision="single",
)

In [ ]:
alm_file = h5py.File(s.alm_file, "r", swmr=True, locking=False)
alms = alm_file["alm"]
fnls = alm_file["fnl"]

In [ ]:
def alm_step_loader(idx):
    print("\ridx: {} ".format(idx), end="")
    return data.compute_alm_sim(s.lensing)


alm_strs_i = np.arange(100)
ksw.step_batch(alm_step_loader, alm_strs_i, comm, verbose=False)

In [ ]:
fisher = float(ksw.compute_fisher())
fisher

In [ ]:
def compute_icov_ell(N, b):
    S_ell = cosmo._camb_data.get_cmb_power_spectra(
        cosmo.camb_params,
        lmax=s.lmax,
        spectra=["total"],
        CMB_unit="muK",
        raw_cl=True,
    )["total"][:, 0]
    b_inv = 1 / b
    return (1 / (S_ell + b_inv * N * b_inv))[None, :]


icov_ell = compute_icov_ell(s.noise_ell, s.beam_ell)
fisher_iso = ksw.compute_fisher_isotropic(icov_ell, comm=comm)
fisher_iso

In [ ]:
def alm_loader(str_idx):
    """Loads in a single alm given a int in string form. Used inside the KSW code."""
    idx, pol = np.unravel_index(int(str_idx), (s.nsims, s.npol))
    print("idx: {}/{}".format(idx, s.nsims))
    return np.array(alms[idx, pol])


alm_strs = np.arange(s.total_sims).astype(str)
estimates = ksw.compute_estimate_batch(
    alm_loader, alm_strs[:100], comm, verbose=False, fisher=fisher
)

In [ ]:
plot_ksw_predictions(fnls[:100, 0], estimates, fisher)